# Huấn luyện Residual PPO trên Kaggle

Tạo một dataset riêng tư từ `rl_kaggle_source.zip`. Import notebook này và đính kèm dataset qua **Add Input**. Chọn CPU bằng **Accelerator → None** và bật Internet để notebook tự tạo môi trường Python 3.12 và cài thư viện. [Tài liệu Kaggle Notebooks](https://www.kaggle.com/docs/notebooks)

Chạy cấu hình `smoke` để kiểm tra cài đặt, đo thời gian và thử lưu checkpoint. Sau khi smoke đạt, chọn `training` cho lượt 51.200 bước cùng ba scenario validation. Mặc định occupancy là `125.0` theo kết quả trung bình của phép so sánh nội bộ với seed 42. Dùng **Save & Run All** để lưu checkpoint, log và kết quả đánh giá dưới `/kaggle/working/smart_buildings_rl/`.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from tempfile import gettempdir
import time
import tomllib
from zipfile import ZipFile

PROFILE = os.environ.get("RL_KAGGLE_PROFILE", "smoke")
OCCUPANCY_SCALE = float(os.environ.get("RL_KAGGLE_OCCUPANCY_SCALE", "125.0"))
SOURCE_INPUT = os.environ.get("RL_KAGGLE_SOURCE")  # Optional explicit path.
INPUT_ROOT = Path(os.environ.get("RL_KAGGLE_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("RL_KAGGLE_OUTPUT_ROOT", "/kaggle/working"))
REUSE_PYTHON = os.environ.get("RL_KAGGLE_REUSE_PYTHON") == "1"

TRAIN_SCENARIO = ("building_train_1", "fairbanks_1")
VALIDATION_SCENARIOS = [
    ("building_train_1", "phoenix_5"),
    ("building_train_2", "fairbanks_2"),
    ("building_train_4", "singapore_3"),
]
profiles = {
    "smoke": (2048, 288, [TRAIN_SCENARIO]),
    "training": (51200, 2016, VALIDATION_SCENARIOS),
}
default_timesteps, default_eval_steps, eval_scenarios = profiles[PROFILE]
timesteps = int(os.environ.get("RL_KAGGLE_TIMESTEPS", default_timesteps))
eval_steps = int(os.environ.get("RL_KAGGLE_EVAL_STEPS", default_eval_steps))
episode_steps = 2016

process_env = {
    **os.environ,
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "NUMBA_NUM_THREADS": "1",
    "MPLBACKEND": "Agg",
    "PYTHONIOENCODING": "utf-8",
    "PYTHONUNBUFFERED": "1",
}
process_env.pop("PYTHONHOME", None)
process_env.pop("PYTHONPATH", None)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
scratch_root = Path(os.environ.get("RL_KAGGLE_SCRATCH_ROOT", gettempdir()))
session_dir = scratch_root / f"smart_buildings_rl_{stamp}"
repo = session_dir / "source"
run_dir = OUTPUT_ROOT.resolve() / "smart_buildings_rl" / f"{PROFILE}-{stamp}"

In [ ]:
if SOURCE_INPUT:
    source = Path(SOURCE_INPUT)
else:
    inputs = sorted(INPUT_ROOT.rglob("rl_kaggle_source.zip"))
    if not inputs:
        inputs = sorted(p.parent for p in INPUT_ROOT.rglob("KAGGLE_SOURCE.json"))
    if len(inputs) != 1:
        raise RuntimeError(f"Attach one RL source bundle or set SOURCE_INPUT: {inputs}")
    source = inputs[0]

if source.is_dir():
    shutil.copytree(source, repo)
else:
    with ZipFile(source) as archive:
        archive.extractall(repo)

manifest_bytes = (repo / "KAGGLE_SOURCE.json").read_bytes()
manifest = json.loads(manifest_bytes)
for name, expected in manifest["files"].items():
    path = repo / name
    if not path.is_file() or sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Source bundle file missing or changed: {name}")

print(f"Source base commit: {manifest['base_commit']}")
print(f"Verified {len(manifest['files'])} source files.")

In [ ]:
if REUSE_PYTHON:
    python = Path(sys.executable)
else:
    setup_env = {**process_env, "UV_CACHE_DIR": str(session_dir / "uv-cache")}
    setup_env.setdefault("UV_PYTHON_INSTALL_DIR", str(session_dir / "python"))
    uv_binary = shutil.which("uv")
    if uv_binary:
        uv = [uv_binary]
    else:
        bootstrap = session_dir / "uv-bootstrap"
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-cache-dir",
             "--target", str(bootstrap), "uv==0.13.10"],
            env=process_env, check=True,
        )
        setup_env["PYTHONPATH"] = str(bootstrap)
        uv = [sys.executable, "-m", "uv"]

    print(f"Notebook Python: {sys.version.split()[0]}. Preparing Python 3.12 for training.", flush=True)
    environment = session_dir / "venv"
    subprocess.run(uv + ["venv", "--python", "3.12", str(environment)], env=setup_env, check=True)
    python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    lock = tomllib.loads((repo / "uv.lock").read_text(encoding="utf-8"))
    pins = {p["name"]: p["version"] for p in lock["package"]}
    torch_pin = f"torch=={pins['torch']}+cpu"
    install = uv + ["pip", "install", "--python", str(python)]
    subprocess.run(
        install + ["--index-url", "https://download.pytorch.org/whl/cpu", torch_pin],
        cwd=repo,
        env=setup_env,
        check=True,
    )
    subprocess.run(
        install + ["--requirements", "pyproject.toml", "--extra", "rl",
                   torch_pin, f"stable-baselines3=={pins['stable-baselines3']}"],
        cwd=repo,
        env=setup_env,
        check=True,
    )

packages = ("numpy", "numba", "gymnasium", "torch", "stable-baselines3")
versions = subprocess.check_output(
    [str(python), "-c",
     "import json, platform, sys; from importlib.metadata import version; "
     "assert sys.version_info[:2] == (3, 12), sys.version; "
     "print(json.dumps({'python': platform.python_version(), "
     "**{name: version(name) for name in sys.argv[1:]}}))",
     *packages],
    cwd=repo,
    env=process_env,
    text=True,
)
print(versions.strip())

In [ ]:
if timesteps <= 0 or timesteps % 1024:
    raise ValueError("TIMESTEPS must be a positive multiple of PPO's 1,024-step rollout.")
if not 1 <= eval_steps <= 8640:
    raise ValueError("EVAL_STEPS must be between 1 and 8,640.")
run_dir.mkdir(parents=True)
config = {
    "profile": PROFILE,
    "source_base_commit": manifest["base_commit"],
    "source_manifest_sha256": sha256(manifest_bytes).hexdigest(),
    "runtime": json.loads(versions),
    "device": "cpu",
    "training_scenario": "/".join(TRAIN_SCENARIO),
    "evaluation_scenarios": ["/".join(pair) for pair in eval_scenarios],
    "seed": 42,
    "episode_steps": episode_steps,
    "timesteps": timesteps,
    "evaluation_steps": eval_steps,
    "occupancy_scale": OCCUPANCY_SCALE,
}
config_path = run_dir / "run_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

started = time.perf_counter()
subprocess.run(
    [str(python), "-m", "scripts.train_ppo",
     "--building", TRAIN_SCENARIO[0], "--weather", TRAIN_SCENARIO[1],
     "--timesteps", str(timesteps), "--episode-steps", str(episode_steps),
     "--occupancy-scale", str(OCCUPANCY_SCALE), "--name", str(run_dir / "training")],
    cwd=repo,
    env=process_env,
    check=True,
)
config["train_elapsed_seconds"] = time.perf_counter() - started
config["train_steps_per_second"] = timesteps / config["train_elapsed_seconds"]
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
checkpoint = run_dir / "training/final_model.zip"
print(f"Checkpoint: {checkpoint}")
print(f"Training throughput including setup: {config['train_steps_per_second']:.2f} steps/s")

In [ ]:
comparisons = []
for building, weather in eval_scenarios:
    subprocess.run(
        [str(python), "-m", "scripts.evaluate_ppo",
         "--building", building, "--weather", weather, "--steps", str(eval_steps),
         "--checkpoint", str(checkpoint), "--name", str(run_dir / "evaluation")],
        cwd=repo,
        env=process_env,
        check=True,
    )
    path = run_dir / "evaluation" / building / weather / "comparison.json"
    comparison = json.loads(path.read_text(encoding="utf-8"))
    if any(comparison[policy]["total_steps"] != eval_steps for policy in ("pid", "ppo")):
        raise RuntimeError(f"Unexpected rollout length: {path}")
    comparisons.append(comparison)
    print(f"{building}/{weather}: PPO minus PID = {comparison['reward_improvement']:+.6f}")

summary = {
    "mean_reward_improvement_per_step": (
        sum(result["reward_improvement"] for result in comparisons)
        / len(comparisons) / eval_steps
    ),
    "scenarios": comparisons,
}
(run_dir / "validation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved outputs in {run_dir}")